In [19]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
# 配置参数，对图片进行处理
class Config:
    data_root = r"C:\Users\28405\CUB_200_2011\CUB_200_2011"
    images_dir = os.path.join(data_root, "images")  # 图片根目录
    split_file = os.path.join(data_root, "train_test_split.txt")  # 训练/测试划分
    classes_file = os.path.join(data_root, "classes.txt") 
    images_file = os.path.join(data_root, "images.txt") 
    
    # 超参数
    img_size = 224
    batch_size = 32  
    epochs = 1
    lr = 1e-4
    weight_decay = 1e-5
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    save_path = "best_bird_model.pth"

cfg = Config()

# 进行数据加载与预处理，从classes.txt读取类别映射：{文件夹名: 数字标签(0-199)} 
def load_class_mapping():
    class_map = {}
    with open(cfg.classes_file, "r", encoding="utf-8") as f:
        lines = f.readlines()
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # 拆分：行号（1-200），我下载的数据集里的文件夹名（如001.Black_footed_Albatross）
        line_num, folder_name = line.split(" ", 1)
        # 标签从0开始（行号1→0，行号200→199）
        class_map[folder_name] = int(line_num) - 1
    return class_map
#读取文件，返回训练/验证/测试集路径列表，从images.txt获取image_id→路径映射，将数据结合train_test_split.txt划分
def load_train_test_split():
    img_id_to_path = {}
    with open(cfg.images_file, "r", encoding="utf-8") as f:
        lines = f.readlines()
    for line in lines:
        line = line.strip()
        if not line:
            continue
        img_id, img_rel_path = line.split(" ", 1)  # 命名规则是img_rel_path: 001.Black_footed_Albatross/xxx.jpg
        img_id_to_path[img_id] = img_rel_path
    
    # 读取训练/测试划分
    train_paths = []
    test_paths = []
    with open(cfg.split_file, "r", encoding="utf-8") as f:
        lines = f.readlines()
    for line in lines:
        line = line.strip()
        if not line:
            continue
        img_id, is_train = line.split(" ", 1)
        if img_id in img_id_to_path:
            img_abs_path = os.path.join(cfg.images_dir, img_id_to_path[img_id])
            if is_train == "1":  # 1=训练集，0=测试集
                train_paths.append(img_abs_path)
            else:
                test_paths.append(img_abs_path)
    
    # 从训练集划分10%作为验证集（8:1:1）
    train_paths, val_paths = train_test_split(train_paths, test_size=0.1, random_state=42)
    return train_paths, val_paths, test_paths

class BirdDataset(Dataset):
    def __init__(self, img_paths, class_map, transform=None):
        self.img_paths = img_paths
        self.class_map = class_map
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path).convert("RGB")
        # 从路径提取文件夹名（如001.Black_footed_Albatross）
        folder_name = os.path.basename(os.path.dirname(img_path))
        label = self.class_map[folder_name]
        if self.transform:
            image = self.transform(image)
        return image, label
#数据预处理：训练集增强，测试集仅标准化
def get_data_transforms():
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(cfg.img_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_test_transform = transforms.Compose([
        transforms.Resize((cfg.img_size + 32, cfg.img_size + 32)),
        transforms.CenterCrop(cfg.img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_test_transform
#创建训练/验证/测试DataLoader
def create_dataloaders():
    class_map = load_class_mapping()
    train_paths, val_paths, test_paths = load_train_test_split()
    train_transform, val_test_transform = get_data_transforms()
    
    train_dataset = BirdDataset(train_paths, class_map, train_transform)
    val_dataset = BirdDataset(val_paths, class_map, val_test_transform)
    test_dataset = BirdDataset(test_paths, class_map, val_test_transform)
    
    # 构建DataLoader（Windows下num_workers=0避免报错）
    train_loader = DataLoader(
        train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True
    )
    
    print(f"数据集划分完成：")
    print(f"训练集：{len(train_dataset)}张图像")
    print(f"验证集：{len(val_dataset)}张图像")
    print(f"测试集：{len(test_dataset)}张图像")
    return train_loader, val_loader, test_loader

# 模型构建
def build_model(num_classes=200):
    model = models.resnet18(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    # 修改全连接层
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    for param in model.layer4.parameters():
        param.requires_grad = True
    for param in model.fc.parameters():
        param.requires_grad = True
    return model.to(cfg.device)

# 训练验证函数 
def train_one_epoch(model, train_loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(cfg.device), labels.to(cfg.device)
        
        # 前向传播，我们在损失函数传入outputs和labels
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 反向传播+优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 统计指标
        total_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy
#验证/测试一个epoch
def validate(model, val_loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # 禁用梯度计算
        for images, labels in tqdm(val_loader, desc="Validation"):
            images, labels = images.to(cfg.device), labels.to(cfg.device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)  # 这里我们修正：传入labels
            
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

# 主训练流程 
def train_model():
    # 创建数据加载器
    train_loader, val_loader, test_loader = create_dataloaders()
    
    # 构建模型、损失函数、优化器
    model = build_model(num_classes=200)
    criterion = nn.CrossEntropyLoss().to(cfg.device)
    optimizer = optim.AdamW(
        model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=3, factor=0.5, verbose=True
    )
    
    # 训练循环
    best_val_acc = 0.0
    for epoch in range(1, cfg.epochs + 1):
        print(f"\n===== Epoch {epoch}/{cfg.epochs} =====")
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}")
        val_loss, val_acc = validate(model, val_loader, criterion)
        print(f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}")
        scheduler.step(val_acc)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), cfg.save_path)
            print(f"Best model saved! Current best val accuracy: {best_val_acc:.4f}")
    
    # 测试集评估
    print(f"\n===== Testing Best Model =====")
    model.load_state_dict(torch.load(cfg.save_path))
    test_loss, test_acc = validate(model, test_loader, criterion)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")
    print(f"训练完成！最佳模型已保存至：{cfg.save_path}")

# 运行训练
if __name__ == "__main__":
    train_model()

数据集划分完成：
训练集：5394张图像
验证集：600张图像
测试集：5794张图像

===== Epoch 1/30 =====


Training: 100%|██████████████████████████████████████████████████████████████████████| 169/169 [07:59<00:00,  2.84s/it]


Train Loss: 4.6323, Train Accuracy: 0.1277


Validation: 100%|██████████████████████████████████████████████████████████████████████| 19/19 [00:41<00:00,  2.20s/it]


Val Loss: 3.7001, Val Accuracy: 0.2817
Best model saved! Current best val accuracy: 0.2817

===== Epoch 2/30 =====


...

Train Loss: 0.0253, Train Accuracy: 0.9994


Validation: 100%|██████████████████████████████████████████████████████████████████████| 19/19 [00:27<00:00,  1.46s/it]


Val Loss: 1.1724, Val Accuracy: 0.6967

===== Testing Best Model =====


Validation: 100%|██████████████████████████████████████████████████████████████████| 182/182 [3:51:11<00:00, 76.22s/it]

Test Loss: 1.2388, Test Accuracy: 0.6869
训练完成！最佳模型已保存至：best_bird_model.pth
